In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

/Users/martindufour/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# Function 3
print('Function 3')
func3_inputs = np.load('./initial_data/function_3/initial_inputs.npy')
print(func3_inputs)

func3_outputs = np.load('./initial_data/function_3/initial_outputs.npy')
print(func3_outputs)
print('/n')

Function 3
[[0.17152521 0.34391687 0.2487372 ]
 [0.24211446 0.64407427 0.27243281]
 [0.53490572 0.39850092 0.17338873]
 [0.49258141 0.61159319 0.34017639]
 [0.13462167 0.21991724 0.45820622]
 [0.34552327 0.94135983 0.26936348]
 [0.15183663 0.43999062 0.99088187]
 [0.64550284 0.39714294 0.91977134]
 [0.74691195 0.28419631 0.22629985]
 [0.17047699 0.6970324  0.14916943]
 [0.22054934 0.29782524 0.34355534]
 [0.66601366 0.67198515 0.2462953 ]
 [0.04680895 0.23136024 0.77061759]
 [0.60009728 0.72513573 0.06608864]
 [0.96599485 0.86111969 0.56682913]]
[-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837]
/n


In [3]:
# Week 1 Input and Output Data
week_1_inputs = [ np.array([0.155793, 0.528435]), np.array([0.224164, 0.812385]), np.array([0.830919, 0.158523, 0.528406]), np.array([0.912533, 0.052672, 0.771239, 0.219812]), np.array([0.234189, 0.83648 , 0.884484, 0.873516]), np.array([0.490808, 0.618683, 0.277824, 0.900494, 0.106596]), np.array([0.067896, 0.486672, 0.255422, 0.215118, 0.427428, 0.72097 ]), np.array([0.061447, 0.062956, 0.029929, 0.036786, 0.407935, 0.795055, 0.496307,0.888085]) ]
week_1_outputs = [np.float64(1.5311489892413605e-58), np.float64(0.04614596805685454), np.float64(-0.04591165123945737), np.float64(-24.387232512869755), np.float64(1049.4420694211206), np.float64(-0.849252655626155), np.float64(1.3793294734939503), np.float64(9.598780741169)]

week_2_inputs = [np.array([0.946399, 0.18731 ]), np.array([0.712637, 0.921564]), np.array([0.765104, 0.052672, 0.438597]), np.array([0.380965, 0.771701, 0.089479, 0.536968]), np.array([0.219189, 0.85148 , 0.874484, 0.883516]), np.array([0.114755, 0.697421, 0.354179, 0.887624, 0.589139]), np.array([0.070896, 0.484672, 0.259422, 0.216118, 0.427428, 0.72297 ]), np.array([0.062447, 0.061956, 0.030929, 0.035786, 0.408935, 0.794055, 0.497307, 0.887085])]
week_2_outputs = [np.float64(2.338449488843798e-206), np.float64(0.5728372778652475), np.float64(-0.10235480701941752), np.float64(-13.368584815829887), np.float64(1109.9883069580462), np.float64(-1.492909868264592), np.float64(1.3944037721068683), np.float64(9.598978827169)]

week_3_inputs = [np.array([0.583738, 0.706798]), np.array([0.699637, 0.928564]), np.array([0.123457, 0.876543, 0.5     ]), np.array([0.230785, 0.914568, 0.102938, 0.657322]), np.array([0.214189, 0.85648 , 0.869484, 0.888516]), np.array([0.051235, 0.987654, 0.43211 , 0.123457, 0.765432]), np.array([0.071896, 0.483672, 0.261422, 0.217118, 0.426428, 0.72497 ]), np.array([0.063447, 0.060956, 0.031929, 0.034786, 0.409935, 0.793055, 0.498307, 0.886085])]
week_3_outputs = [np.float64(9.817718646044271e-07), np.float64(0.49978778689465564), np.float64(-0.04054152149606349), np.float64(-21.112987453792396), np.float64(1132.5255136709882), np.float64(-2.4679805862566795), np.float64(1.4059619293543582), np.float64(9.599155713169)]

week_4_inputs = [np.array([0.867072, 0.913241]), np.array([0.83604 , 0.696071]), np.array([0.447658, 0.395195, 0.505344]), np.array([0.422706, 0.385497, 0.37529 , 0.410833]), np.array([0.157706, 0.912326, 0.830158, 0.925715]), np.array([0.275401, 0.      , 0.568079, 1.      , 0.121326]), np.array([0.124679, 0.389027, 0.389012, 0.23686 , 0.379036, 0.802247]), np.array([0.077447, 0.21029 , 0.114032, 0.159535, 0.695924, 0.531316,0.178973, 0.57168 ])]
week_4_outputs = [np.float64(-1.0508862613282513e-96), np.float64(0.20657541488475104), np.float64(-0.03229682155733878), np.float64(0.4964978124935766), np.float64(1445.380906735266), np.float64(-0.8152779914672599), np.float64(2.0217812896063525), np.float64(9.987130922543)]

week_5_inputs = [np.array([0.065052, 0.948886]), np.array([0.388677, 0.271349]), np.array([1.      , 0.136717, 0.850593]), np.array([0.908266, 0.239562, 0.144895, 0.489453]), np.array([0.115614, 0.955641, 0.820859, 0.938174]), np.array([0.568442, 0.      , 1.      , 1.      , 1.      ]), np.array([0.134015, 0.028783, 0.755137, 0.62031 , 0.70408 , 0.212964]), np.array([0.127008, 0.292876, 0.06967 , 0.277582, 0.553407, 0.547258,0.220835, 0.443146])]
week_5_outputs = [np.float64(2.0262778967114778e-283), np.float64(0.016418658339648333), np.float64(-0.054388754089278846), np.float64(-17.161465002411145), np.float64(1779.8600577462366), np.float64(-1.9906490554141107), np.float64(0.052467603616080494), np.float64(9.9149654065019)]

week_6_inputs = [np.array([0.17701 , 0.088703]), np.array([0.593592, 0.679102]), np.array([0.7536, 1.    , 1.    ]), np.array([0.370348, 0.379959, 0.430216, 0.444351]), np.array([0.169493, 0.556801, 0.936155, 0.69603 ]), np.array([0.366464, 0.316099, 1.      , 1.      , 0.      ]), np.array([0.123574, 0.270452, 0.482301, 0.208163, 0.330398, 0.872733]), np.array([0.3191  , 0.828915, 0.037008, 0.59627 , 0.230009, 0.120567,0.076953, 0.696289])]
week_6_outputs = [np.float64(-2.3725238219366144e-119), np.float64(0.06836721478932847), np.float64(-0.48310415434111403), np.float64(0.18076540708623456), np.float64(282.83820524691396), np.float64(-0.8214281898088153), np.float64(2.075605759888563), np.float64(8.8803427965834)]

week_7_inputs = [np.array([0.065052, 0.948886]), np.array([0.140924, 0.802197]), np.array([1., 1., 0.]), np.array([0.32078 , 0.186519, 0.040775, 0.590893]), np.array([0.548734, 0.691895, 0.651961, 0.224269]), np.array([0.368433, 0.      , 1.      , 1.      , 0.428022]), np.array([0.139689, 0.317433, 0.463608, 0.250431, 0.325485, 0.810756]), np.array([0.      , 0.202823, 0.231703, 0.      , 1.      , 1.      ,
       0.288474, 1.      ])]
week_7_outputs = [np.float64(2.0262778967114778e-283), np.float64(-0.10826299524356352), np.float64(-0.16354530625442043), np.float64(-11.58523458824008), np.float64(1.9931553503870212), np.float64(-1.2796687884296385), np.float64(2.462201676843452), np.float64(9.622023932692)]

week_8_inputs = [np.array([0.000788, 0.033717]), np.array([0.914607, 0.789979]), np.array([0.317253, 0.002183, 0.963506]), np.array([0.008334, 0.234163, 0.946857, 0.993453]), np.array([0.078217, 0.973099, 0.868006, 0.933352]), np.array([0.426158, 0.348959, 0.616644, 0.692851, 0.024814]), np.array([0.269889, 0.346609, 0.467584, 0.256632, 0.302654, 0.800092]), np.array([0.066269, 0.029193, 0.143059, 0.209133, 0.840619, 0.604919,
       0.230297, 0.701468])]
week_8_outputs = [np.float64(1.5608341712501477e-228), np.float64(0.0347797753016137), np.float64(-0.3756702789549372), np.float64(-33.661790988299735), np.float64(2151.3700669834334), np.float64(-0.23000336822278494), np.float64(2.4516632535923746), np.float64(9.9644234438351)]

In [4]:
# Function 3
print('Function 3')
# Load inputs from previous run
# Loads initial data
week_0_func3_inputs = np.load('./initial_data/function_3/initial_inputs.npy')
week_0_func3_outputs = np.load('./initial_data/function_3/initial_outputs.npy')

print(f'Shape of initial input data: {week_0_func3_inputs.shape}')
print(f'Shape of initial output data: {week_0_func3_outputs.shape}')

print(f'Week 1 inputs: {week_1_inputs[2]}')
print(f'Week 2 inputs: {week_2_inputs[2]}')
print(f'Week 3 inputs: {week_3_inputs[2]}')
print(f'Week 4 inputs: {week_4_inputs[2]}')
print(f'Week 5 inputs: {week_5_inputs[2]}')
print(f'Week 6 inputs: {week_6_inputs[2]}')
print(f'Week 7 inputs: {week_7_inputs[2]}')
print(f'Week 8 inputs: {week_8_inputs[2]}')

combined_func3_inputs = np.vstack([
    week_0_func3_inputs,
    week_1_inputs[2],
    week_2_inputs[2],
    week_3_inputs[2],
    week_4_inputs[2],
    week_5_inputs[2],
    week_6_inputs[2],
    week_7_inputs[2],
    week_8_inputs[2]
])
print(f'Number of input data points: {len(combined_func3_inputs)}')
print('Combined input data')
print(combined_func3_inputs)

# Load outputs from previous run
week_func3_output = week_1_outputs[2]
combined_func3_outputs = np.concatenate([
    week_0_func3_outputs,
    [week_1_outputs[2]],
    [week_2_outputs[2]],
    [week_3_outputs[2]],
    [week_4_outputs[2]],
    [week_5_outputs[2]],
    [week_6_outputs[2]],
    [week_7_outputs[2]],
    [week_8_outputs[2]]
])
print(f'Number of output data points: {len(combined_func3_outputs)}')
print('Combined output data')
print(combined_func3_outputs)

Function 3
Shape of initial input data: (15, 3)
Shape of initial output data: (15,)
Week 1 inputs: [0.830919 0.158523 0.528406]
Week 2 inputs: [0.765104 0.052672 0.438597]
Week 3 inputs: [0.123457 0.876543 0.5     ]
Week 4 inputs: [0.447658 0.395195 0.505344]
Week 5 inputs: [1.       0.136717 0.850593]
Week 6 inputs: [0.7536 1.     1.    ]
Week 7 inputs: [1. 1. 0.]
Week 8 inputs: [0.317253 0.002183 0.963506]
Number of input data points: 23
Combined input data
[[0.17152521 0.34391687 0.2487372 ]
 [0.24211446 0.64407427 0.27243281]
 [0.53490572 0.39850092 0.17338873]
 [0.49258141 0.61159319 0.34017639]
 [0.13462167 0.21991724 0.45820622]
 [0.34552327 0.94135983 0.26936348]
 [0.15183663 0.43999062 0.99088187]
 [0.64550284 0.39714294 0.91977134]
 [0.74691195 0.28419631 0.22629985]
 [0.17047699 0.6970324  0.14916943]
 [0.22054934 0.29782524 0.34355534]
 [0.66601366 0.67198515 0.2462953 ]
 [0.04680895 0.23136024 0.77061759]
 [0.60009728 0.72513573 0.06608864]
 [0.96599485 0.86111969 0.566829

In [8]:
### ====== OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 3 ======
# Import the OptunaBayesianOptimizer class
import sys
sys.path.insert(0, './bayesian_optimization_challenge-md')
from bo_optuna import OptunaBayesianOptimizer

# Create optimizer instance with initial Function 3 data
print("=" * 60)
print("OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 3")
print("=" * 60)

X_func3_initial = week_0_func3_inputs
y_func3_initial = week_0_func3_outputs
bounds_func3 = [(0, 1), (0, 1), (0, 1)]

optimizer_func3 = OptunaBayesianOptimizer(
    X_initial=X_func3_initial,
    y_initial=y_func3_initial,
    bounds=bounds_func3,
    optimize_hp=True,  # Enable hyperparameter tuning
    random_state=42,
    acquisition="ucb"
)

print(f"\nInitial training data shape: X={optimizer_func3.X_train.shape}, y={optimizer_func3.y_train.shape}")
print(f"Initial best observation: {optimizer_func3.get_best_observation()[1]:.6e}")


OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 3

Initial training data shape: X=(15, 3), y=(15,)
Initial best observation: -3.483531e-02


In [10]:
# Run Optuna-based BO for 8 weeks (8 iterations)
print("\nRunning Optuna-based BO for 8 iterations...")
print("-" * 60)

# Get all weekly data
weekly_data = [
    (week_1_inputs[2], week_1_outputs[2]),
    (week_2_inputs[2], week_2_outputs[2]),
    (week_3_inputs[2], week_3_outputs[2]),
    (week_4_inputs[2], week_4_outputs[2]),
    (week_5_inputs[2], week_5_outputs[2]),
    (week_6_inputs[2], week_6_outputs[2]),
    (week_7_inputs[2], week_7_outputs[2]),
    (week_8_inputs[2], week_8_outputs[2])
]

optuna_proposals_func3 = []
manual_best_func3 = week_0_func3_outputs.max()
optuna_best_func3 = y_func3_initial.max()

for week, (x_actual, y_actual) in enumerate(weekly_data, start=1):
    print(f"\nWeek {week}:")
    print(f"  Actual observation: y = {y_actual:.6e}")
    
    # Get Optuna proposal
    proposals = optimizer_func3.optimize(
        n_iterations=1,
        optimize_hp_every=1 if week % 2 == 0 else 0,  # Tune HP every other week
        optimize_hp_n_trials=30,
        acq_n_trials=100,
        verbose=True
    )
    
    x_proposed = proposals[0]
    optuna_proposals_func3.append(x_proposed)
    
    # Update optimizer with actual observation
    optimizer_func3.update(x_actual, y_actual)
    
    # Track best values
    manual_best_func3 = max(manual_best_func3, y_actual)
    optuna_best_func3 = max(optuna_best_func3, y_actual)
    
    print(f"  Best so far (Optuna): {optuna_best_func3:.6e}")

print("\n" + "=" * 60)
print("OPTUNA-BASED BO COMPLETED FOR FUNCTION 3")
print("=" * 60)


[I 2026-04-15 04:18:36,039] A new study created in memory with name: no-name-e3b5d0d1-3c2e-461a-9d71-c35327bd0143
[I 2026-04-15 04:18:36,042] Trial 0 finished with value: 1.9886720847742065 and parameters: {'x0': 0.3745401188473625, 'x1': 0.9507143064099162, 'x2': 0.7319939418114051}. Best is trial 0 with value: 1.9886720847742065.
[I 2026-04-15 04:18:36,044] Trial 1 finished with value: 1.0855661908216052 and parameters: {'x0': 0.5986584841970366, 'x1': 0.15601864044243652, 'x2': 0.15599452033620265}. Best is trial 0 with value: 1.9886720847742065.
[I 2026-04-15 04:18:36,046] Trial 2 finished with value: 0.8019604898715005 and parameters: {'x0': 0.05808361216819946, 'x1': 0.8661761457749352, 'x2': 0.6011150117432088}. Best is trial 0 with value: 1.9886720847742065.
[I 2026-04-15 04:18:36,050] Trial 3 finished with value: 1.7893290984018582 and parameters: {'x0': 0.7080725777960455, 'x1': 0.020584494295802447, 'x2': 0.9699098521619943}. Best is trial 0 with value: 1.9886720847742065.
[


Running Optuna-based BO for 8 iterations...
------------------------------------------------------------

Week 1:
  Actual observation: y = -4.591165e-02


[I 2026-04-15 04:18:36,109] Trial 12 finished with value: 0.8456412333974639 and parameters: {'x0': 0.9627289272173465, 'x1': 0.04140532812059714, 'x2': 0.7836296211490168}. Best is trial 0 with value: 1.9886720847742065.
[I 2026-04-15 04:18:36,125] Trial 13 finished with value: 1.0499477845079417 and parameters: {'x0': 0.6833819300429151, 'x1': 0.4647683830492125, 'x2': 0.7473941873052088}. Best is trial 0 with value: 1.9886720847742065.
[I 2026-04-15 04:18:36,139] Trial 14 finished with value: 0.635380908212396 and parameters: {'x0': 0.3142030565443681, 'x1': 0.4774479680075873, 'x2': 0.9906170268211734}. Best is trial 0 with value: 1.9886720847742065.
[I 2026-04-15 04:18:36,148] Trial 15 finished with value: 1.570685284939625 and parameters: {'x0': 0.6195339431335991, 'x1': 0.960893749374683, 'x2': 0.7676259977007913}. Best is trial 0 with value: 1.9886720847742065.
[I 2026-04-15 04:18:36,157] Trial 16 finished with value: 1.1078513290938534 and parameters: {'x0': 0.9076931568042685

[Iteration 0] Proposed: [0.04757376 0.99832512 0.99912491], UCB: 2.603678
  Best so far (Optuna): -3.483531e-02

Week 2:
  Actual observation: y = -1.023548e-01


[I 2026-04-15 04:18:37,482] Trial 6 finished with value: 0.5928309394182103 and parameters: {'x0': 0.43194501864211576, 'x1': 0.2912291401980419, 'x2': 0.6118528947223795}. Best is trial 0 with value: 1.988673721546814.
[I 2026-04-15 04:18:37,484] Trial 7 finished with value: 0.22002506187353407 and parameters: {'x0': 0.13949386065204183, 'x1': 0.29214464853521815, 'x2': 0.3663618432936917}. Best is trial 0 with value: 1.988673721546814.
[I 2026-04-15 04:18:37,486] Trial 8 finished with value: 0.4726851681175641 and parameters: {'x0': 0.45606998421703593, 'x1': 0.7851759613930136, 'x2': 0.19967378215835974}. Best is trial 0 with value: 1.988673721546814.
[I 2026-04-15 04:18:37,489] Trial 9 finished with value: 0.5832859297637063 and parameters: {'x0': 0.5142344384136116, 'x1': 0.5924145688620425, 'x2': 0.046450412719997725}. Best is trial 0 with value: 1.988673721546814.
[I 2026-04-15 04:18:37,499] Trial 10 finished with value: 1.6424365329409023 and parameters: {'x0': 0.31134533481454

...

[Iteration 0] Proposed: [0.04757376 0.99832512 0.99912491], UCB: 2.603682
  Best so far (Optuna): -3.229682e-02

Week 7:
  Actual observation: y = -1.635453e-01


[I 2026-04-15 04:18:44,398] Trial 5 finished with value: 0.5271280095191122 and parameters: {'x0': 0.18340450985343382, 'x1': 0.3042422429595377, 'x2': 0.5247564316322378}. Best is trial 0 with value: 1.9886925980183385.
[I 2026-04-15 04:18:44,400] Trial 6 finished with value: 0.5927236573334504 and parameters: {'x0': 0.43194501864211576, 'x1': 0.2912291401980419, 'x2': 0.6118528947223795}. Best is trial 0 with value: 1.9886925980183385.
[I 2026-04-15 04:18:44,404] Trial 7 finished with value: 0.22003216799319605 and parameters: {'x0': 0.13949386065204183, 'x1': 0.29214464853521815, 'x2': 0.3663618432936917}. Best is trial 0 with value: 1.9886925980183385.
[I 2026-04-15 04:18:44,408] Trial 8 finished with value: 0.47269571562576523 and parameters: {'x0': 0.45606998421703593, 'x1': 0.7851759613930136, 'x2': 0.19967378215835974}. Best is trial 0 with value: 1.9886925980183385.
[I 2026-04-15 04:18:44,412] Trial 9 finished with value: 0.5833038164351404 and parameters: {'x0': 0.51423443841

[Iteration 0] Proposed: [0.04757376 0.99832512 0.99912491], UCB: 2.603695
  Best so far (Optuna): -3.229682e-02

Week 8:
  Actual observation: y = -3.756703e-01


[I 2026-04-15 04:18:45,896] Trial 11 finished with value: 0.08982845062893136 and parameters: {'x0': 0.7688315225052019, 'x1': 0.9838906689434096, 'x2': 0.9343591819013546}. Best is trial 0 with value: 1.9886927630260147.
[I 2026-04-15 04:18:45,915] Trial 12 finished with value: 0.8456100923298394 and parameters: {'x0': 0.9627289272173465, 'x1': 0.04140532812059714, 'x2': 0.7836296211490168}. Best is trial 0 with value: 1.9886927630260147.
[I 2026-04-15 04:18:45,928] Trial 13 finished with value: 1.0499522735011486 and parameters: {'x0': 0.6833819300429151, 'x1': 0.4647683830492125, 'x2': 0.7473941873052088}. Best is trial 0 with value: 1.9886927630260147.
[I 2026-04-15 04:18:45,946] Trial 14 finished with value: 0.6354011860610074 and parameters: {'x0': 0.3142030565443681, 'x1': 0.4774479680075873, 'x2': 0.9906170268211734}. Best is trial 0 with value: 1.9886927630260147.
[I 2026-04-15 04:18:45,957] Trial 15 finished with value: 1.5706980637597254 and parameters: {'x0': 0.619533943133

[Iteration 0] Proposed: [0.04757376 0.99832512 0.99912491], UCB: 2.603695
  Best so far (Optuna): -3.229682e-02

OPTUNA-BASED BO COMPLETED FOR FUNCTION 2
